In [11]:
import pandas as pd

xl = pd.ExcelFile("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Heat_Flow_DB/IHFC_2024_GHFDB_v.2026.03.xlsx")
print(f"Sheets: {xl.sheet_names}")

df = pd.read_excel("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Heat_Flow_DB/IHFC_2024_GHFDB_v.2026.03.xlsx",sheet_name=xl.sheet_names[1],header=5)
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(df.head())

Sheets: ['Metadata', 'GHFDB R20024 v.2026.03']
Shape: (91182, 67)

Columns: ['q', 'q_uncertainty', 'name', 'lat_NS', 'long_EW', 'elevation', 'environment', 'p_comment', 'corr_HP_flag', 'total_depth_MD', 'total_depth_TVD', 'explo_method', 'explo_purpose', 'qc', 'qc_uncertainty', 'q_method', 'q_top', 'q_bottom', 'probe_penetration', 'publication_reference', 'data_reference', 'relevant_child', 'c_comment', 'corr_IS_flag', 'corr_T_flag', 'corr_S_flag', 'corr_E_flag', 'corr_TOPO_flag', 'corr_PAL_flag', 'corr_SUR_flag', 'corr_CONV_flag', 'corr_HR_flag', 'expedition', 'probe_type', 'probe_length', 'probe_tilt', 'water_temperature', 'geo_lithology', 'geo_stratigraphy', 'T_grad_mean', 'T_grad_uncertainty', 'T_grad_mean_cor', 'T_grad_uncertainty_cor', 'T_method_top', 'T_method_bottom', 'T_shutin_top', 'T_shutin_bottom', 'T_corr_top', 'T_corr_bottom', 'T_number', 'q_date', 'tc_mean', 'tc_uncertainty', 'tc_source', 'tc_location', 'tc_method', 'tc_saturation', 'tc_pT_conditions', 'tc_pT_fuction', '

In [12]:
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)

Shape: (91182, 67)

Columns: ['q', 'q_uncertainty', 'name', 'lat_NS', 'long_EW', 'elevation', 'environment', 'p_comment', 'corr_HP_flag', 'total_depth_MD', 'total_depth_TVD', 'explo_method', 'explo_purpose', 'qc', 'qc_uncertainty', 'q_method', 'q_top', 'q_bottom', 'probe_penetration', 'publication_reference', 'data_reference', 'relevant_child', 'c_comment', 'corr_IS_flag', 'corr_T_flag', 'corr_S_flag', 'corr_E_flag', 'corr_TOPO_flag', 'corr_PAL_flag', 'corr_SUR_flag', 'corr_CONV_flag', 'corr_HR_flag', 'expedition', 'probe_type', 'probe_length', 'probe_tilt', 'water_temperature', 'geo_lithology', 'geo_stratigraphy', 'T_grad_mean', 'T_grad_uncertainty', 'T_grad_mean_cor', 'T_grad_uncertainty_cor', 'T_method_top', 'T_method_bottom', 'T_shutin_top', 'T_shutin_bottom', 'T_corr_top', 'T_corr_bottom', 'T_number', 'q_date', 'tc_mean', 'tc_uncertainty', 'tc_source', 'tc_location', 'tc_method', 'tc_saturation', 'tc_pT_conditions', 'tc_pT_fuction', 'tc_number', 'tc_strategy', 'Ref_ISGN', 'Year', 

In [13]:
# Basic stats on heat flow values
q = pd.to_numeric(df['q'], errors='coerce')
print(f"\nHeat flow (q) stats:")
print(f"  Valid values:  {q.notna().sum():,} of {len(q):,}")
print(f"  Range:         {q.min():.1f} – {q.max():.1f} mW/m²")
print(f"  Mean:          {q.mean():.1f} mW/m²")
print(f"  Median:        {q.median():.1f} mW/m²")
print(f"  Negative vals: {(q < 0).sum()}")
print(f"  > 500 mW/m²:   {(q > 500).sum()} (likely anomalous)")

# Coverage per patch
patches = {
    "Kanto_Japan":      (138.5, 34.5, 141.5, 37.2),
    "Tohoku_Japan":     (140.5, 37.5, 143.5, 40.5),
    "Central_Chile":    (-72.5, -36.5, -69.5, -33.5),
    "Central_Turkey":   (35.5, 36.5, 39.0, 39.0),
    "Nepal":            (83.5, 27.0, 86.5, 29.7),
    "North_Island_NZ":  (174.5, -40.5, 178.0, -37.5),
    "Sumatra":          (100.5, -5.5, 104.5, -2.0),
    "Kutch_India":      (68.5, 21.5, 72.0, 24.5),
    "Sichuan_China":    (102.0, 29.5, 105.5, 32.5),
    "W_Australia":      (117.0, -32.0, 120.5, -29.0),
    "S_Norway":         (5.0, 58.5, 9.0, 61.5),
    "Ordos_China":      (107.5, 37.0, 111.0, 40.0),
}

lat = pd.to_numeric(df['lat_NS'], errors='coerce')
lon = pd.to_numeric(df['long_EW'], errors='coerce')
df['q_num'] = q
df['lat_num'] = lat
df['lon_num'] = lon

print(f"\n{'Patch':<25} {'N points':>9} {'Min':>8} {'Max':>8} {'Mean':>8} {'NaN q':>8}")
print("-" * 65)
for name, (minlon, minlat, maxlon, maxlat) in patches.items():
    patch = df[
        (df['lon_num'] >= minlon) & (df['lon_num'] <= maxlon) &
        (df['lat_num'] >= minlat) & (df['lat_num'] <= maxlat)
    ]
    q_patch = patch['q_num']
    valid = q_patch.dropna()
    print(f"{name:<25} {len(patch):>9} {valid.min() if len(valid)>0 else float('nan'):>8.1f} "
          f"{valid.max() if len(valid)>0 else float('nan'):>8.1f} "
          f"{valid.mean() if len(valid)>0 else float('nan'):>8.1f} "
          f"{q_patch.isna().sum():>8}")


Heat flow (q) stats:
  Valid values:  91,182 of 91,182
  Range:         -6120.0 – 489000.0 mW/m²
  Mean:          198.9 mW/m²
  Median:        66.0 mW/m²
  Negative vals: 89
  > 500 mW/m²:   2975 (likely anomalous)

Patch                      N points      Min      Max     Mean    NaN q
-----------------------------------------------------------------
Kanto_Japan                     205     15.0   2020.0    107.4        0
Tohoku_Japan                    190     11.0    571.0    100.4        0
Central_Chile                    30     30.0    335.0    110.6        0
Central_Turkey                   28     33.0    201.0     78.4        0
Nepal                             0      nan      nan      nan        0
North_Island_NZ                 374   -107.0  48000.0    682.2        0
Sumatra                          46     29.8    128.3     98.1        0
Kutch_India                       0      nan      nan      nan        0
Sichuan_China                    36     33.0     70.5     54.6       

In [14]:
# Apply quality filter
q = pd.to_numeric(df['q'], errors='coerce')
lat = pd.to_numeric(df['lat_NS'], errors='coerce')
lon = pd.to_numeric(df['long_EW'], errors='coerce')

df['q_num'] = q
df['lat_num'] = lat
df['lon_num'] = lon

# Filter to physically plausible range
# 0–500 mW/m² covers everything from stable craton to active geothermal
df_clean = df[
    (df['q_num'] > 0) &
    (df['q_num'] <= 500) &
    df['lat_num'].notna() &
    df['lon_num'].notna()
].copy()

print(f"Raw:     {len(df):,} points")
print(f"Cleaned: {len(df_clean):,} points ({100*len(df_clean)/len(df):.1f}% retained)")
print(f"Removed: {len(df)-len(df_clean):,} points")
print(f"\nCleaned q stats:")
print(f"  Range:  {df_clean['q_num'].min():.1f} – {df_clean['q_num'].max():.1f} mW/m²")
print(f"  Mean:   {df_clean['q_num'].mean():.1f} mW/m²")
print(f"  Median: {df_clean['q_num'].median():.1f} mW/m²")

# Re-check patches with expanded radius for Nepal and Kutch
patches_expanded = {
    "Nepal":       (81.0, 25.0, 89.0, 32.0),   # 4 deg buffer
    "Kutch_India": (66.0, 19.0, 74.0, 27.0),   # 3 deg buffer
}

print(f"\nExpanded search for empty patches:")
for name, (minlon, minlat, maxlon, maxlat) in patches_expanded.items():
    patch = df_clean[
        (df_clean['lon_num'] >= minlon) & (df_clean['lon_num'] <= maxlon) &
        (df_clean['lat_num'] >= minlat) & (df_clean['lat_num'] <= maxlat)
    ]
    print(f"  {name}: {len(patch)} points in expanded bounds "
          f"(mean={patch['q_num'].mean():.1f} mW/m²)" if len(patch) > 0 
          else f"  {name}: still 0 points even with buffer")

# Re-run patch stats on cleaned data
patches = {
    "Kanto_Japan":      (138.5, 34.5, 141.5, 37.2),
    "Tohoku_Japan":     (140.5, 37.5, 143.5, 40.5),
    "Central_Chile":    (-72.5, -36.5, -69.5, -33.5),
    "Central_Turkey":   (35.5, 36.5, 39.0, 39.0),
    "Nepal":            (83.5, 27.0, 86.5, 29.7),
    "North_Island_NZ":  (174.5, -40.5, 178.0, -37.5),
    "Sumatra":          (100.5, -5.5, 104.5, -2.0),
    "Kutch_India":      (68.5, 21.5, 72.0, 24.5),
    "Sichuan_China":    (102.0, 29.5, 105.5, 32.5),
    "W_Australia":      (117.0, -32.0, 120.5, -29.0),
    "S_Norway":         (5.0, 58.5, 9.0, 61.5),
    "Ordos_China":      (107.5, 37.0, 111.0, 40.0),
}

print(f"\n{'Patch':<25} {'N points':>9} {'Min':>8} {'Max':>8} {'Mean':>8}")
print("-" * 60)
for name, (minlon, minlat, maxlon, maxlat) in patches.items():
    patch = df_clean[
        (df_clean['lon_num'] >= minlon) & (df_clean['lon_num'] <= maxlon) &
        (df_clean['lat_num'] >= minlat) & (df_clean['lat_num'] <= maxlat)
    ]
    q_p = patch['q_num']
    if len(q_p) > 0:
        print(f"{name:<25} {len(q_p):>9} {q_p.min():>8.1f} "
              f"{q_p.max():>8.1f} {q_p.mean():>8.1f}")
    else:
        print(f"{name:<25} {'NO DATA':>9}")

Raw:     91,182 points
Cleaned: 88,084 points (96.6% retained)
Removed: 3,098 points

Cleaned q stats:
  Range:  0.1 – 500.0 mW/m²
  Mean:   84.3 mW/m²
  Median: 65.0 mW/m²

Expanded search for empty patches:
  Nepal: still 0 points even with buffer
  Kutch_India: 101 points in expanded bounds (mean=76.3 mW/m²)

Patch                      N points      Min      Max     Mean
------------------------------------------------------------
Kanto_Japan                     202     15.0    479.0     87.6
Tohoku_Japan                    189     11.0    410.0     97.9
Central_Chile                    30     30.0    335.0    110.6
Central_Turkey                   28     33.0    201.0     78.4
Nepal                       NO DATA
North_Island_NZ                 319      3.0    498.0    110.3
Sumatra                          46     29.8    128.3     98.1
Kutch_India                 NO DATA
Sichuan_China                    36     33.0     70.5     54.6
W_Australia                      12     27.0     

## Insights

<p>The IHFC Global Heat Flow Database was delivered as an Excel file containing 91,182 measurement points with 67 attribute columns covering heat flow values, measurement metadata, quality scores, and geographic coordinates. The raw dataset required careful preprocessing. The first five rows contain metadata and unit headers rather than data, and the actual column names (q, lat_NS, long_EW) are embedded in row 4. After correct header parsing, the full dataset was available for inspection.</p>

<p>Raw heat flow values span an implausible range of -6,120 to 489,000 mW/m², immediately indicating the presence of outliers, unit inconsistencies, and hydrothermal vent measurements that are not representative of crustal heat flow. A quality filter retaining only physically plausible values between 0 and 500 mW/m² removed 3,098 points (3.4%) and produced a cleaned dataset of 88,084 points. After cleaning, the global range is 0.1–500 mW/m² with a mean of 84.3 mW/m² and median of 65 mW/m². These are both consistent with published estimates for mean continental heat flow. New Zealand illustrates the impact of the cleaning most clearly: the raw patch mean of 682 mW/m² (driven by values up to 48,000 mW/m² from hydrothermal measurements in the Taupo Volcanic Zone) reduces to a physically meaningful 110.3 mW/m² after filtering, correctly reflecting the genuinely elevated but bounded heat flow of an active volcanic arc.</p>

<p>Patch-level means after cleaning are physically ordered as expected and provide useful discriminative signal for the geological clustering step. New Zealand and Chile record the highest means (110.3 and 110.6 mW/m² respectively), reflecting the Taupo Volcanic Zone and Andean volcanism as dominant heat sources. The subduction patches — Tohoku (97.9 mW/m²) and Kanto (87.6 mW/m²) — show elevated but more moderate values consistent with subduction-related heat transfer. The stable craton and shield patches record the lowest means: Western Australia (46.8 mW/m²), Norway (43.3 mW/m²), and Ordos (62.7 mW/m²), all consistent with old, cold lithosphere with low radiogenic heat production. Sichuan (54.6 mW/m²) is notable — lower than expected for a tectonically active setting, likely reflecting the insulating effect of the thick sedimentary basin cover dampening surface heat flow measurements.</p>

<p>Two patches have no heat flow data within their bounding boxes. Nepal returns zero points even after expanding the search radius to a 4-degree buffer — a genuine data absence reflecting the extreme difficulty of heat flow measurement in high-altitude Himalayan terrain. Heat flow will be excluded from the Nepal feature vector and replaced with a binary heat_flow_available = 0 mask flag during processing. Kutch returns zero points within bounds but 101 points in the expanded buffer region with a mean of 76.3 mW/m², sufficient for kriging-based interpolation into the patch during processing. Both patches will be documented in the Methods section with explicit treatment strategies, and an ablation excluding imputed heat flow cells will be run to verify that the imputation does not materially affect clustering or transfer results.</p>

<p>A lower bound concern also exists for Chile (minimum 30 mW/m²) and Tohoku (minimum 11 mW/m²), where suspiciously low values likely represent deep offshore measurements with poor quality control that survived the 500 mW/m² upper filter. A lower bound of 20 mW/m² will be applied during processing to remove these. For patches where heat flow coverage remains insufficient after cleaning, literature-based imputation using published regional estimates is a defensible and standard approach — Nepal will receive a literature-derived value of approximately 70–80 mW/m² based on Tanaka et al. (2004) and ICDP borehole studies, with an imputation flag retained throughout the analysis.</p>

<p>Overall the heat flow layer is the sparsest and noisiest of the static layers inspected so far, reflecting the inherent difficulty of subsurface thermal measurement at global scale. Despite this, the cleaned patch-level means are physically interpretable and directionally consistent with the geological regime classifications established by the other layers. The combination of high heat flow in volcanic and subduction settings and low heat flow in stable cratons provides a complementary thermal dimension to the structural and compositional signals captured by Vs30, sediment thickness, crustal thickness, and DEM.</p>